In [14]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import glob
import warnings
warnings.filterwarnings('ignore')

# Pour l'affichage dans le notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

In [15]:
# Définir le chemin vers le dossier data
data_path = "./data/"

# Lister tous les fichiers CSV dans le dossier
csv_files = glob.glob(os.path.join(data_path, "*.csv"))
print(f"Fichiers CSV trouvés dans {data_path}:")
for file in csv_files:
    print(f"  - {os.path.basename(file)}")

Fichiers CSV trouvés dans ./data/:
  - bds_posts_hot.csv
  - bds_posts_top.csv
  - boycott_israel_posts_hot.csv
  - boycott_israel_posts_top.csv


In [16]:
# Charger chaque fichier avec inspection préalable
def load_and_inspect_csv(file_path):
    """Charge un fichier CSV et retourne des informations de base"""
    filename = os.path.basename(file_path)
    print(f"\n{'='*60}")
    print(f"Chargement de: {filename}")
    print(f"{'='*60}")
    
    # Lire le fichier
    df = pd.read_csv(file_path)
    
    # Afficher les informations de base
    print(f"Dimensions: {df.shape[0]} lignes × {df.shape[1]} colonnes")
    print(f"Colonnes: {list(df.columns)}")
    print(f"Plage temporelle (created_utc): {df['created_utc'].min()} à {df['created_utc'].max()}")
    print(f"Valeurs manquantes: {df.isnull().sum().sum()}")
    
    # Afficher les premières lignes
    print(f"\nAperçu des données (3 premières lignes):")
    print(df.head(3))
    
    return df, filename

In [17]:
# Charger tous les fichiers
dfs = {}
for file_path in csv_files:
    df, filename = load_and_inspect_csv(file_path)
    dfs[filename] = df


Chargement de: bds_posts_hot.csv
Dimensions: 175 lignes × 24 colonnes
Colonnes: ['id', 'subreddit', 'permalink', 'title', 'selftext', 'created_utc_formatted', 'created_utc', 'author', 'score', 'ups', 'downs', 'num_comments', 'upvote_ratio', 'author_fullname', 'subreddit_subscribers', 'total_awards_received', 'link_flair_text', 'over_18', 'locked', 'created_local_formatted', 'created_local', 'edited', 'selftext_length', 'downs_calculated']
Plage temporelle (created_utc): 1739156288.0 à 1768851948.0
Valeurs manquantes: 195

Aperçu des données (3 premières lignes):
        id subreddit                                          permalink                                              title                                           selftext created_utc_formatted   created_utc        author  score  ups  downs  num_comments  upvote_ratio author_fullname  subreddit_subscribers  total_awards_received link_flair_text  over_18  locked created_local_formatted  created_local edited  selftext_length  

In [18]:
def clean_dataframe(df, filename):
    """Nettoie et prépare un dataframe"""
    
    # Créer une copie pour éviter les modifications inplace
    df_clean = df.copy()
    
    # Extraire le nom du subreddit et le type (hot/top) du nom du fichier
    # Format attendu: boycott_israel_posts_hot.csv ou bds_posts_top.csv
    name_parts = filename.replace('.csv', '').split('_')
    
    if 'boycott' in filename.lower() and 'israel' in filename.lower():
        subreddit_name = 'BoycottIsrael'
    elif 'bds' in filename.lower():
        subreddit_name = 'BDS'
    else:
        subreddit_name = 'Unknown'
    
    # Déterminer le type (hot ou top)
    if 'hot' in filename.lower():
        post_type = 'hot'
    elif 'top' in filename.lower():
        post_type = 'top'
    else:
        post_type = 'unknown'
    
    # Ajouter des colonnes pour identification
    df_clean['source_file'] = filename
    df_clean['subreddit'] = subreddit_name
    df_clean['post_type'] = post_type
    
    # Convertir les colonnes de date
    if 'created_utc' in df_clean.columns:
        df_clean['created_utc'] = pd.to_datetime(df_clean['created_utc'], unit='s', errors='coerce')
    
    if 'created_local' in df_clean.columns:
        # Essayer de parser la date locale - format peut varier
        try:
            df_clean['created_local'] = pd.to_datetime(df_clean['created_local'], errors='coerce')
        except:
            print(f"Attention: Problème avec la conversion de created_local dans {filename}")
    
    # Nettoyer les colonnes de texte
    text_columns = ['title', 'selftext', 'link_flair_text']
    for col in text_columns:
        if col in df_clean.columns:
            # Remplacer les NaN par des chaînes vides
            df_clean[col] = df_clean[col].fillna('')
            # Convertir en string
            df_clean[col] = df_clean[col].astype(str)
    
    # S'assurer que les colonnes numériques sont bien numériques
    numeric_columns = ['score', 'ups', 'downs', 'num_comments', 'upvote_ratio', 
                       'total_awards_received', 'selftext_length', 'downs_calculated']
    for col in numeric_columns:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    # Vérifier et calculer downs_calculated si nécessaire
    if 'downs_calculated' in df_clean.columns and 'ups' in df_clean.columns and 'score' in df_clean.columns:
        # Vérifier si la colonne downs_calculated est vide ou incorrecte
        if df_clean['downs_calculated'].isnull().all() or (df_clean['downs_calculated'] == 0).all():
            # Calculer les downs à partir de ups et score: downs = ups - score
            df_clean['downs_calculated'] = df_clean['ups'] - df_clean['score']
            df_clean['downs_calculated'] = df_clean['downs_calculated'].clip(lower=0)  # éviter les valeurs négatives
    
    return df_clean

In [19]:
# Appliquer le nettoyage à tous les dataframes
cleaned_dfs = {}
for filename, df in dfs.items():
    print(f"\nNettoyage de {filename}...")
    cleaned_df = clean_dataframe(df, filename)
    cleaned_dfs[filename] = cleaned_df
    
    # Afficher quelques informations après nettoyage
    print(f"  Subreddit: {cleaned_df['subreddit'].iloc[0]}")
    print(f"  Type de post: {cleaned_df['post_type'].iloc[0]}")
    print(f"  Dates converties: {cleaned_df['created_utc'].min().date()} à {cleaned_df['created_utc'].max().date()}")


Nettoyage de bds_posts_hot.csv...
  Subreddit: BDS
  Type de post: hot
  Dates converties: 2025-02-10 à 2026-01-19

Nettoyage de bds_posts_top.csv...
  Subreddit: BDS
  Type de post: top
  Dates converties: 2024-05-01 à 2026-01-01

Nettoyage de boycott_israel_posts_hot.csv...
  Subreddit: BoycottIsrael
  Type de post: hot
  Dates converties: 2025-09-25 à 2026-01-19

Nettoyage de boycott_israel_posts_top.csv...
  Subreddit: BoycottIsrael
  Type de post: top
  Dates converties: 2024-11-27 à 2026-01-16


In [20]:
# Fusionner tous les dataframes en un seul
print("\n" + "="*60)
print("FUSION DES DATAFRAMES")
print("="*60)

# Identifier les colonnes communes à tous les dataframes
common_columns = None
for filename, df in cleaned_dfs.items():
    if common_columns is None:
        common_columns = set(df.columns)
    else:
        common_columns = common_columns.intersection(set(df.columns))

print(f"Colonnes communes à tous les dataframes ({len(common_columns)}):")
print(sorted(list(common_columns)))


FUSION DES DATAFRAMES
Colonnes communes à tous les dataframes (26):
['author', 'author_fullname', 'created_local', 'created_local_formatted', 'created_utc', 'created_utc_formatted', 'downs', 'downs_calculated', 'edited', 'id', 'link_flair_text', 'locked', 'num_comments', 'over_18', 'permalink', 'post_type', 'score', 'selftext', 'selftext_length', 'source_file', 'subreddit', 'subreddit_subscribers', 'title', 'total_awards_received', 'ups', 'upvote_ratio']


In [21]:
# Créer une liste de tous les dataframes avec seulement les colonnes communes
dataframes_to_merge = []
for filename, df in cleaned_dfs.items():
    df_common = df[list(common_columns)].copy()
    dataframes_to_merge.append(df_common)
    print(f"{filename}: {len(df_common)} lignes")

# Fusionner tous les dataframes
combined_df = pd.concat(dataframes_to_merge, ignore_index=True)

print(f"\nDataframe combiné: {combined_df.shape[0]} lignes × {combined_df.shape[1]} colonnes")

bds_posts_hot.csv: 175 lignes
bds_posts_top.csv: 100 lignes
boycott_israel_posts_hot.csv: 125 lignes
boycott_israel_posts_top.csv: 101 lignes

Dataframe combiné: 501 lignes × 26 colonnes


In [22]:
def analyze_combined_data(df):
    """Analyse approfondie du dataframe combiné"""
    
    print("\n" + "="*60)
    print("ANALYSE DU DATAFRAME COMBINÉ")
    print("="*60)
    
    # 1. Répartition par subreddit et type de post
    print("\n1. RÉPARTITION PAR SUBREDDIT ET TYPE DE POST:")
    distribution = df.groupby(['subreddit', 'post_type']).size().reset_index(name='count')
    print(distribution)
    
    # 2. Statistiques temporelles
    print(f"\n2. PÉRIODE COUVERTE:")
    print(f"   Début: {df['created_utc'].min()}")
    print(f"   Fin: {df['created_utc'].max()}")
    print(f"   Durée: {(df['created_utc'].max() - df['created_utc'].min()).days} jours")
    
    # 3. Statistiques par subreddit
    print("\n3. STATISTIQUES PAR SUBREDDIT:")
    for subreddit in df['subreddit'].unique():
        sub_df = df[df['subreddit'] == subreddit]
        print(f"\n   {subreddit}:")
        print(f"   - Nombre de posts: {len(sub_df)}")
        print(f"   - Score moyen: {sub_df['score'].mean():.1f}")
        print(f"   - Score max: {sub_df['score'].max()}")
        print(f"   - Nombre moyen de commentaires: {sub_df['num_comments'].mean():.1f}")
        print(f"   - Période: {sub_df['created_utc'].min().date()} à {sub_df['created_utc'].max().date()}")
    
    # 4. Valeurs manquantes
    print("\n4. VALEURS MANQUANTES PAR COLONNE:")
    missing_data = df.isnull().sum()
    missing_percent = (missing_data / len(df)) * 100
    missing_df = pd.DataFrame({
        'Valeurs manquantes': missing_data,
        'Pourcentage': missing_percent
    })
    print(missing_df[missing_df['Valeurs manquantes'] > 0].sort_values('Pourcentage', ascending=False))
    
    # 5. Auteurs uniques
    print(f"\n5. NOMBRE D'AUTEURS UNIQUES: {df['author'].nunique()}")
    
    # 6. Analyse des colonnes de texte
    print("\n6. ANALYSE DES COLONNES DE TEXTE:")
    text_cols = ['title', 'selftext', 'link_flair_text']
    for col in text_cols:
        if col in df.columns:
            non_empty = df[col].astype(str).str.strip() != ''
            empty_count = len(df) - non_empty.sum()
            print(f"   {col}: {empty_count} valeurs vides ({empty_count/len(df)*100:.1f}%)")
    
    return distribution

In [23]:
# Exécuter l'analyse
distribution = analyze_combined_data(combined_df)


ANALYSE DU DATAFRAME COMBINÉ

1. RÉPARTITION PAR SUBREDDIT ET TYPE DE POST:
       subreddit post_type  count
0            BDS       hot    175
1            BDS       top    100
2  BoycottIsrael       hot    125
3  BoycottIsrael       top    101

2. PÉRIODE COUVERTE:
   Début: 2024-05-01 13:10:55
   Fin: 2026-01-19 19:45:48
   Durée: 628 jours

3. STATISTIQUES PAR SUBREDDIT:

   BDS:
   - Nombre de posts: 275
   - Score moyen: 329.7
   - Score max: 1147
   - Nombre moyen de commentaires: 13.6
   - Période: 2024-05-01 à 2026-01-19

   BoycottIsrael:
   - Nombre de posts: 226
   - Score moyen: 183.4
   - Score max: 750
   - Nombre moyen de commentaires: 11.3
   - Période: 2024-11-27 à 2026-01-19

4. VALEURS MANQUANTES PAR COLONNE:
                 Valeurs manquantes  Pourcentage
selftext_length                 253    50.499002
author_fullname                  14     2.794411

5. NOMBRE D'AUTEURS UNIQUES: 255

6. ANALYSE DES COLONNES DE TEXTE:
   title: 0 valeurs vides (0.0%)
   selftext

In [25]:
# Définir le chemin de sauvegarde
output_path = "./data/cleaned/"
os.makedirs(output_path, exist_ok=True)

# Sauvegarder le dataframe combiné
combined_file = os.path.join(output_path, "reddit_boycott_combined.csv")
combined_df.to_csv(combined_file, index=False, encoding='utf-8')
print(f"\nDataframe combiné sauvegardé dans: {combined_file}")

# Sauvegarder également en format pickle pour préserver les types de données
pickle_file = os.path.join(output_path, "reddit_boycott_combined.pkl")
combined_df.to_pickle(pickle_file)
print(f"Dataframe combiné sauvegardé (pickle) dans: {pickle_file}")


Dataframe combiné sauvegardé dans: ./data/cleaned/reddit_boycott_combined.csv
Dataframe combiné sauvegardé (pickle) dans: ./data/cleaned/reddit_boycott_combined.pkl


In [26]:
# Créer un résumé statistique final
print("\n" + "="*60)
print("RÉSUMÉ FINAL DE LA PHASE DE NETTOYAGE ET FUSION")
print("="*60)

print(f"\n📊 DONNÉES COMBINÉES:")
print(f"   • Total de posts: {len(combined_df):,}")
print(f"   • Subreddits: {', '.join(combined_df['subreddit'].unique())}")
print(f"   • Types de posts: {', '.join(combined_df['post_type'].unique())}")

print(f"\n📅 COUVERTURE TEMPORELLE:")
print(f"   • Période: {combined_df['created_utc'].min().strftime('%d/%m/%Y')} au {combined_df['created_utc'].max().strftime('%d/%m/%Y')}")
print(f"   • Durée: {(combined_df['created_utc'].max() - combined_df['created_utc'].min()).days} jours")

print(f"\n👥 DÉMOGRAPHIE:")
print(f"   • Auteurs uniques: {combined_df['author'].nunique():,}")
print(f"   • Ratio posts/auteur: {len(combined_df)/combined_df['author'].nunique():.1f}")

print(f"\n📈 ENGAGEMENT MOYEN:")
print(f"   • Score moyen: {combined_df['score'].mean():.1f}")
print(f"   • Commentaires moyens par post: {combined_df['num_comments'].mean():.1f}")
print(f"   • Ratio upvote moyen: {combined_df['upvote_ratio'].mean():.2f}")

print(f"\n💾 FICHIERS CRÉÉS:")
print(f"   1. {combined_file} ({os.path.getsize(combined_file)/1024/1024:.1f} MB)")
print(f"   2. {pickle_file} ({os.path.getsize(pickle_file)/1024/1024:.1f} MB)")

print(f"\n✅ PHASE 1 TERMINÉE AVEC SUCCÈS!")
print(f"   Le dataframe est prêt pour l'analyse exploratoire (Phase 2).")


RÉSUMÉ FINAL DE LA PHASE DE NETTOYAGE ET FUSION

📊 DONNÉES COMBINÉES:
   • Total de posts: 501
   • Subreddits: BDS, BoycottIsrael
   • Types de posts: hot, top

📅 COUVERTURE TEMPORELLE:
   • Période: 01/05/2024 au 19/01/2026
   • Durée: 628 jours

👥 DÉMOGRAPHIE:
   • Auteurs uniques: 255
   • Ratio posts/auteur: 2.0

📈 ENGAGEMENT MOYEN:
   • Score moyen: 263.7
   • Commentaires moyens par post: 12.5
   • Ratio upvote moyen: 0.98

💾 FICHIERS CRÉÉS:
   1. ./data/cleaned/reddit_boycott_combined.csv (0.3 MB)
   2. ./data/cleaned/reddit_boycott_combined.pkl (0.3 MB)

✅ PHASE 1 TERMINÉE AVEC SUCCÈS!
   Le dataframe est prêt pour l'analyse exploratoire (Phase 2).
